# Sprint 6: Policy dashboard

Review policy actions for Kibera, Mathare, and Mukuru. The signed-in dashboard shows a policy map, parcel details, and explanations using the five GraphSAGE features. The Sprint 5 river-corridor lift was 0.929. An official validation against the 37 NRRP zones is still pending.

Use the Drive artifacts from the same Sprint 5 run so the table rows and graph nodes represent the same parcels.


## 0. Setup

Open this notebook in Colab. Setup checks out GitHub's main branch and installs the dashboard dependencies.


In [1]:
%%bash
set -e
REPO=https://github.com/emmanuelamor5/nairobi-risk-mapping.git
CHECKOUT=/content/nairobi-risk-mapping
if [ -d "$CHECKOUT/.git" ]; then
  cd "$CHECKOUT"
  git fetch origin
  git checkout main
  git merge --ff-only origin/main
else
  git clone --branch main "$REPO" "$CHECKOUT"
fi


Your branch is up to date with 'origin/main'.
Already up to date.


Already on 'main'


In [2]:
%cd /content/nairobi-risk-mapping
!pip install -q -r requirements.txt


/content/nairobi-risk-mapping


## 1. Gate and explanations

A password-only token is refused. A token that records `sign_in_second_factor` is accepted. The checks below use claim dicts, not a live Firebase project.


In [3]:
import torch
from torch_geometric.data import Data

from src.dashboard.auth_gate import mfa_satisfied
from src.models.attention_unet_model import AttentionUNetModel
from src.models.graphsage_model import GraphSAGEModel

assert not mfa_satisfied({"firebase": {"sign_in_provider": "password"}})
assert mfa_satisfied({"firebase": {"sign_in_second_factor": "totp"}})

graph = Data(
    x=torch.randn(8, 5),
    edge_index=torch.tensor([[0, 1, 2], [1, 2, 0]], dtype=torch.long),
    y=torch.randint(0, 3, (8,)),
)
sage = GraphSAGEModel(in_channels=5, hidden_channels=8, num_classes=3, dropout=0.0)
report = sage.explain(graph, node_index=0)
print("SHAP features:", report["ranked_features"])
print("attributions:", [round(float(v), 4) for v in report["attributions"]])

unet = AttentionUNetModel(in_channels=9, base_channels=4, device="cpu")
heat = unet.explain(torch.rand(1, 9, 32, 32))
print("Grad-CAM", tuple(heat.shape), "range", round(float(heat.min()), 3), round(float(heat.max()), 3))


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


SHAP features: ['dist_to_road_m', 'roof_material_score', 'dist_to_water_m', 'dist_to_sanitation_m', 'area_m2']
attributions: [-0.0035, 0.0077, -0.0069, 0.0571, -0.0268]
Grad-CAM (32, 32) range 0.0 1.0


## 2. Verify and join the Drive inputs

The Colab project folder is `MyDrive/nairobi_risk_mapping/`. Sprint 5's `policy_table.npz` is directly inside that folder. The three `{settlement}_graph.pt` files are under `data/graphs/`.

The cells below mount Drive and join the table to the graph positions. They print the selected paths and parcel count.


In [4]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
drive_project = Path("/content/drive/MyDrive/nairobi_risk_mapping")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import numpy as np

from src.dashboard.app import _load_records
from src.dashboard.map_view import build_policy_map
from src.models.fusion import resolve_graphs_dir

table_path = drive_project / "policy_table.npz"
if not table_path.is_file():
    raise FileNotFoundError(f"Expected the Sprint 5 policy table at {table_path}")

graphs_dir = resolve_graphs_dir([drive_project / "data"])
expected_graphs_dir = drive_project / "data" / "graphs"
if graphs_dir != expected_graphs_dir:
    raise FileNotFoundError(f"Expected the Sprint 6 graphs at {expected_graphs_dir}")

records = _load_records(str(table_path), str(graphs_dir))
keys = [str(key) for key in records["settlement_keys"]]
print("Joined parcels", records["positions"].shape[0], "CRS", records["pos_crs"])

preview = build_policy_map(
    records["positions"][:2],
    records["action_index"][:2],
    flood_proba=np.asarray(records["flood_proba"])[:2],
    deprivation_class=np.asarray(records["deprivation_class"])[:2],
    settlement_index=np.asarray(records["settlement_index"])[:2],
    settlement_keys=keys,
    pos_crs=records["pos_crs"],
)
print("Map preview centre", preview.location)


policy table: /content/drive/MyDrive/nairobi_risk_mapping/policy_table.npz
graphs: /content/drive/MyDrive/nairobi_risk_mapping/data/graphs
Joined parcels 18005 CRS EPSG:32737
Map preview centre [-1.316223405200628, 36.778909628239845]


## 3. Optional authenticated Colab preview

Enable the intended sign-in methods in your Firebase project. For the password and TOTP flow, enable Email/Password sign-in and TOTP MFA.

Add `FIREBASE_API_KEY`, `FIREBASE_AUTH_DOMAIN`, and `FIREBASE_PROJECT_ID` in Colab Secrets and grant this notebook access. Setting `OPEN_PUBLIC_PREVIEW` to `True` starts the dashboard and opens a temporary public HTTPS tunnel to its sign-in page. Add the preview hostname to Firebase Authorized domains if required for your sign-in flow.

Restart the runtime if it has an earlier preview running. Leave this cell running and open the new HTTPS URL printed here. Stop the cell to stop this preview server.


In [6]:
from google.colab import userdata

def firebase_preview_settings():
    settings = {}
    for name in ("FIREBASE_API_KEY", "FIREBASE_AUTH_DOMAIN", "FIREBASE_PROJECT_ID"):
        try:
            value = userdata.get(name)
        except userdata.SecretNotFoundError as exc:
            raise ValueError(f"Add {name} to Colab Secrets.") from exc
        except userdata.NotebookAccessError as exc:
            raise ValueError(f"Enable Notebook access for {name} in Colab Secrets.") from exc
        if not value or not value.strip():
            raise ValueError(f"Set a nonempty value for {name} in Colab Secrets.")
        settings[name] = value.strip()
    return settings


In [7]:
import os
import subprocess
import sys

OPEN_PUBLIC_PREVIEW = True
if OPEN_PUBLIC_PREVIEW:
    env = {**os.environ, **firebase_preview_settings()}
    env.pop("FIREBASE_WEB_CONFIG", None)
    subprocess.run(
        [
            sys.executable, "-u", "-m", "src.dashboard.app",
            "--table", str(table_path), "--graphs", str(graphs_dir),
            "--port", "8800", "--public-preview",
        ],
        env=env,
        check=True,
    )


KeyboardInterrupt: 